## Setup

In [2]:
import os
import sys

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


In [3]:
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
path = 'data/npt-HK4.gro'
abs_path = os.path.join(project_root, path)

# Extracts data from .gro file into multi-index DataFrame (unsorted)
df_gro, title, num_atoms, box_dimensions = gp.read_gro(abs_path, multiply=10, positions=True, velocities=False) # convert nm to Å  
    
# Checking
# df_gro

In [4]:
from utils.generate_mol_meshes import molecules_to_meshes

mol_meshes = molecules_to_meshes(df_gro, box_dimensions, 
                                 sphere_radius_scale=0.8, 
                                 sphere_subdiv=2, 
                                 bond_radius=0.1,
                                 num_processes=None, context='fork')


print(len(mol_meshes))        # 1501
print(mol_meshes[1])          # trimesh.Trimesh object

Processing 1501 molecules with 8 logical cores: 100%|██████████| 1501/1501 [00:22<00:00, 66.53it/s]


1501
<trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>


## 1. Generate e-centroids

In [5]:
from utils.blocking_algo import calculate_e_centroids

e_centroids = calculate_e_centroids(mol_meshes, df_gro, box_dimensions)


Insert atom id for one molecule.
Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' 
----------------------------------------
You entered: [32-53]
----------------------------------------
Selected atom_id(s): [32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53]


## 2. Generate centroids & radii

In [10]:
from utils.blocking_algo import compute_centroids_and_radii_pbc

centroids, radii = compute_centroids_and_radii_pbc(mol_meshes, box_dimensions)

## 3. Neighbor candidates

In [8]:
import numpy as np
from scipy.spatial import KDTree
import utils.mic_helper as mh

def get_neighbor_candidates(box_dimensions, centroids, k=10):
    """
    Find the k nearest neighbors of each molecule, based on centroid distance.
    
    Parameters
    ----------
    box_dimensions : np.ndarray
        Simulation box_dimensions (Å), shape (3,) for orthorhombic or (3,3) for triclinic
    centroids : dict[int, np.ndarray]
        Dictionary where keys are molecule IDs (int) and values are
        the corresponding centroids (numpy arrays of shape (3,)).
    k : int, optional
        Number of nearest neighbors to return per molecule. Default is 10.

    Returns
    -------
    dict[int, list[tuple[int,float]]]
        Mapping mol_id -> list of (neighbor_id, distance).
    """

    ids = list(centroids.keys())
    coords = np.vstack([centroids[i] for i in ids])
    coords_wrapped = mh.wrap_points(coords, box_dimensions)
    kd = KDTree(coords_wrapped, boxsize=box_dimensions)

    neighbors = {}
    for idx, mol_id in enumerate(ids):
        dists, idxs = kd.query(coords[idx], k=k+1)
        dists, idxs = dists[1:], idxs[1:]
        neighbors[mol_id] = [(ids[j], float(d)) for j, d in zip(idxs, dists)]

    # Remove duplicate pairs (A,B) and (B,A), keep only (A,B) where A < B
    neighbor_candidates = [(min(key, t[0]), max(key, t[0]))
                           for key, value in neighbors.items() for t in value if key < t[0]]
    neighbor_candidates = list(set(neighbor_candidates))
    neighbor_candidates.sort()
    
    return neighbor_candidates

In [11]:
neighbor_candidates = get_neighbor_candidates(box_dimensions, centroids, k=10)

len(neighbor_candidates)

7511

In [ ]:
# Calculate and store distances between neighbor candidate pairs using centroids
neighbor_distances = {}

for i, j in neighbor_candidates:
    c_i = centroids[i]
    c_j = centroids[j]
    dist = mh.mic_distance(c_i, c_j, box_dimensions)
    neighbor_distances[(i, j)] = dist

# Filter neighbor_candidates where distance < 10 Å
filtered_neighbor_candidates = [pair for pair, dist in neighbor_distances.items() if dist < 10.0]

print(f"Number of filtered pairs (<10 Å): {len(filtered_neighbor_candidates)}")
print(filtered_neighbor_candidates[:10])  # Show first 10 filtered pairs

Number of filtered pairs (<10 Å): 2206
[(1, 2), (1, 1308), (2, 4), (2, 18), (2, 704), (3, 258), (3, 704), (3, 1380), (3, 1461), (4, 984)]


## 4. Blocking algorithm

In [7]:
def blocked_by_any(i, j, e_centroids, centroids, radii, mol_meshes, neighbor_candidates):
    """
    Determine if the direct path between two molecule e_centroids is obstructed by any other molecule.

    For a given pair of molecules (i, j), this function checks whether the straight line
    connecting their e_centroids is intersected ("blocked") by any other molecule in the system.
    The check is performed in two steps:
      1. Fast sphere rejection: For each candidate blocking molecule, if its centroid is not
         within its effective radius of the line segment, it is skipped.
      2. Ray-mesh intersection: If the sphere check passes, a ray-mesh intersection test is
         performed to determine if the mesh of the candidate molecule blocks the path.

    Periodic boundary conditions (PBC) are handled using the minimum-image convention.

    Parameters
    ----------
    i, j : int
        IDs of the two molecules to test for a direct connection.
    e_centroids : dict[int, np.ndarray]
        Dictionary where keys are molecule IDs (int) and values are
        the corresponding electron clump centroids (numpy arrays of shape (3,)).
    radii : dict[int, float]
        Dictionary where keys are molecule IDs (int) and values are
        their maximum radii measured from centroid to furthest atom.
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary where keys are molecule IDs (int) and values are
        the corresponding molecular meshes (trimesh.Trimesh objects).

    Returns
    -------
    blocked : bool
        True if the path between i and j is blocked by any other molecule, False otherwise.
    """
    # Map molecule IDs to their index in ids
    ci, cj = e_centroids[i], e_centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return False
    direction = seg_vec / seg_len

    # Get candidate molecule IDs (not indices)
    cand_ids = [t[1] for t in neighbor_candidates if t[0] == i]

    for mol_k in cand_ids:
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return True
        
    return False

In [12]:
# Create the new list
new_list = [tup for tup in neighbor_candidates if tup[0] == 10]

print(new_list)

[(10, 58), (10, 125), (10, 266), (10, 316), (10, 418), (10, 587), (10, 623), (10, 901), (10, 1051), (10, 1153)]


In [13]:
for i in [tup[1] for tup in new_list]:
    if blocked_by_any(10, i, e_centroids, centroids, radii, mol_meshes, neighbor_candidates):
        print("10 and", i, "is blocked")


10 and 266 is blocked
10 and 316 is blocked
10 and 623 is blocked
10 and 901 is blocked


## This not working ;(

In [ ]:
def intersect_points(i, j, e_centroids, centroids, radii, mol_meshes, neighbor_candidates):

    # Map molecule IDs to their index in ids
    ci, cj = e_centroids[i], e_centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return False
    direction = seg_vec / seg_len

    # Get candidate molecule IDs (not indices)
    cand_ids = [t[1] for t in neighbor_candidates if t[0] == i]
    cand_ids.append(i)

    for mol_k in cand_ids:
        locs, index_ray, index_tri = mol_meshes[mol_k].ray.intersects_location(
        ray_origins=ci.reshape(1, 3),
        ray_directions=direction.reshape(1, 3)
        )
        print(f"{(i, j, mol_k)}: {locs}")
        

In [29]:
mol_meshes[10]

<trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>

In [27]:
mh.mic_distance(np.array([21.29090828, 80.18365191, 23.13735048]), np.array([21.8163103,  80.11393547, 23.07614426]), 100)

0.5335296299110694

In [ ]:
for i in [tup[1] for tup in new_list]:
    intersect_points(10, i, e_centroids, centroids, radii, mol_meshes, neighbor_candidates)


(10, 58, 58): [[21.29090828 81.18365191 23.13735048]
 [21.8163103  80.11393547 23.07614426]
 [22.01538643 79.70861725 23.05295307]
 [21.44553132 80.86884001 23.11933782]]
(10, 58, 125): []
(10, 58, 266): []
(10, 58, 316): []
(10, 58, 418): []
(10, 58, 587): []
(10, 58, 623): []
(10, 58, 901): []
(10, 58, 1051): []
(10, 58, 1153): []
(10, 58, 10): [[17.99804697 87.88790456 23.52094934]
 [18.4180827  87.03271346 23.47201767]
 [18.29434514 87.28464265 23.48643236]
 [18.87390404 86.104663   23.41891718]
 [19.08702936 85.6707407  23.39408934]]
(10, 125, 58): []
(10, 125, 125): [[18.27973904 80.59686618 33.99104303]
 [18.19929117 81.81705913 32.34871931]
 [18.20365359 81.7508921  32.4377771 ]
 [18.16629064 82.31759456 31.67502159]
 [18.21443972 81.58729351 32.65797329]
 [18.30062779 80.28003621 34.41748163]
 [18.22891495 81.36774045 32.95348164]
 [18.16636909 82.31640463 31.67662317]]
(10, 125, 266): []
(10, 125, 316): []
(10, 125, 418): []
(10, 125, 587): []
(10, 125, 623): []
(10, 125, 901

In [1]:
import numpy as np

def mic_vector(dx, box):
    """
    Minimum-image displacement. dx can be (...,3) or (3,)
    box can be scalar or (3,)
    """
    box = np.asarray(box, dtype=float)
    # allow scalar box
    if box.shape == ():
        box = np.array([box, box, box], dtype=float)
    return dx - box * np.round(dx / box)

def blocked_by_any_new(i, j,
                       e_centroids,
                       centroids,
                       radii,
                       mol_meshes,
                       neighbor_candidates,
                       box=None,
                       self_block=True,
                       tol=1e-3):
    """
    Determine whether the segment from e_centroids[i] -> e_centroids[j] is blocked
    by any molecule present in neighbor_candidates. Parts of molecule i (self-block)
    are considered blocking if they produce an intersection along the segment
    beyond a small tolerance from the origin.

    Parameters
    ----------
    i, j : int
        molecule ids (endpoints)
    e_centroids : dict[int, np.ndarray]
        electron-clump centroids (shape (3,))
    centroids : dict[int, np.ndarray]
        geometric centroids (shape (3,))
    radii : dict[int, float]
        bounding-sphere radii
    mol_meshes : dict[int, trimesh.Trimesh]
        per-molecule meshes (vertices assumed in same coordinates as centroids)
    neighbor_candidates : list[tuple[int,int]]
        candidate neighbor pairs. Used to select likely blockers.
    box : None or array-like (3,) or scalar
        If given, apply MIC (periodic box lengths) to segment and mesh transforms.
        Units must match centroids/mesh coordinates.
    self_block : bool
        If True, allow molecule i (origin) to block (self-occlusion).
    tol : float
        Small distance tolerance in same units as coordinates (ignore intersections
        with t <= tol to avoid counting origin grazing as a block).

    Returns
    -------
    blocked : bool
    blocker_id : int or None
        id of the molecule that blocks (first found), or None if unblocked
    intersections : list of (np.ndarray) 3D points of intersections on the blocking mesh
    """

    # --- 1. Build the segment (apply MIC if box provided) ---
    ci = np.asarray(e_centroids[i], dtype=float)
    cj = np.asarray(e_centroids[j], dtype=float)

    seg_vec = cj - ci
    if box is not None:
        seg_vec = mic_vector(seg_vec, box)

    seg_len = np.linalg.norm(seg_vec)
    if seg_len <= tol:
        return False, None, []

    direction = seg_vec / seg_len  # unit direction

    # --- 2. Build candidate set of molecule ids to test ---
    # Include any molecule that appears as neighbor candidate with i OR j,
    # plus include i and j themselves (to allow self-blocking).
    cand_set = set()
    for a, b in neighbor_candidates:
        if a == i or a == j:
            cand_set.add(b)
        if b == i or b == j:
            cand_set.add(a)
    # Always consider i and j explicitly (so self-blocking possible)
    cand_set.add(i)
    cand_set.add(j)

    # Convert to list (deterministic order)
    cand_ids = sorted(cand_set)

    # Precompute maximum radius for search heuristic (optional)
    max_radius = max(radii.values()) if len(radii) else 0.0

    # --- 3. Loop candidates with cheap sphere rejection then exact ray test ---
    for mol_k in cand_ids:
        # If user disallows self-block, skip i and j
        if not self_block and mol_k in (i, j):
            continue

        # Fast sphere rejection (use MIC distance between centroids and segment)
        ck = np.asarray(centroids[mol_k], dtype=float)

        # Place ck into the same image as the segment start ci (MIC)
        if box is not None:
            ck_img = ci + mic_vector(ck - ci, box)
        else:
            ck_img = ck.copy()

        # Compute closest point on segment to ck_img
        v = seg_vec
        w = ck_img - ci
        denom = np.dot(v, v)
        if denom == 0.0:
            t = 0.0
        else:
            t = np.dot(w, v) / denom
        t_clamped = float(np.clip(t, 0.0, 1.0))
        closest = ci + t_clamped * v
        dist_closest = np.linalg.norm(ck_img - closest)

        # If sphere doesn't touch the segment (with small safety margin), skip
        if dist_closest > (radii[mol_k] + 1e-8):
            continue

        # --- 4. Exact check: transform mesh vertices into ci-image and ray-intersect ---
        mesh_k = mol_meshes[mol_k]

        # Get vertex positions and faces
        verts = np.asarray(mesh_k.vertices, dtype=float)
        faces = np.asarray(mesh_k.faces, dtype=int)

        # Transform verts so they are in the same periodic image as ci
        if box is not None:
            verts_in_frame = ci + mic_vector(verts - ci, box)
        else:
            verts_in_frame = verts.copy()

        # Build a temporary mesh object with transformed vertices (do not process)
        try:
            temp_mesh = mesh_k.copy()  # copy to preserve original
            temp_mesh.vertices = verts_in_frame
            # skip processing to keep indices consistent
        except Exception:
            # fallback: recreate mesh (less efficient)
            import trimesh
            temp_mesh = trimesh.Trimesh(vertices=verts_in_frame, faces=faces, process=False)

        # Ray intersection: use intersects_location for consistent return
        locs, index_ray, index_tri = temp_mesh.ray.intersects_location(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        )

        if len(locs) == 0:
            # no intersection with this mesh
            continue

        # Compute distances along ray (project onto direction)
        t_vals = np.dot(locs - ci, direction)  # scalar projection (can be negative)
        # Keep only intersections that lie along finite segment (tol < t < seg_len - tol)
        mask = (t_vals > tol) & (t_vals < seg_len - tol)
        valid_points = locs[mask]
        valid_t = t_vals[mask]

        if valid_points.shape[0] > 0:
            # We found intersection(s) along the segment by mol_k (could be i itself)
            # Return first blocker (you may choose smallest t if you want nearest block)
            order = np.argsort(valid_t)
            intersections = [valid_points[idx] for idx in order]
            return True, mol_k, intersections

    # none blocked
    return False, None, []


In [10]:
from utils.blocking_algo import find_neighbor_pairs

neighbor_pairs, e_centroids, centroids, neighbor_candidates = find_neighbor_pairs(mol_meshes, df_gro, box_dimensions, path, k=10, export_csv=True)

print(e_centroids)
print(centroids)
print(neighbor_candidates)
print(neighbor_pairs)

Insert atom id for one molecule.
Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' 
----------------------------------------
You entered: [32-53]
----------------------------------------
Selected atom_id(s): [32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53]


Testing neighbor pairs: 100%|██████████| 7511/7511 [02:41<00:00, 46.58it/s] 


----------------------------------------
Successfully exported npt-HK4_e_centroids.csv
Successfully exported npt-HK4_centroids.csv
Successfully exported npt-HK4_neighbor_candidates.csv
Exported npt-HK4_neighbor_pairs.csv
{1: array([ 15.75727273,   3.93227273, 111.68174545]), 2: array([  4.935     ,   7.26454545, 108.39227273]), 3: array([  7.66909091, 108.07818182,   8.69727273]), 4: array([ 1.52731818, 10.37727273,  1.91959091]), 5: array([55.41954545,  1.17957273, 58.02727273]), 6: array([92.80727273, 66.47681818,  7.53409091]), 7: array([ 68.19590909, 100.06045455,  71.07681818]), 8: array([25.97636364, 68.23909091, 69.13727273]), 9: array([79.10227273, 92.49590909, 84.83954545]), 10: array([17.76818182, 88.35590909, 23.54772727]), 11: array([78.41590909, 97.155     , 21.39      ]), 12: array([73.15636364, 62.76681818, 26.63363636]), 13: array([85.55136364, 65.40545455, 70.38954545]), 14: array([111.43042727,   1.28506364,  47.98727273]), 15: array([105.975     ,  38.25545455,   7.7

In [1]:
len(neighbor_pairs)

NameError: name 'neighbor_pairs' is not defined

## View molecule

In [12]:
import pyglet
import time
import trimesh

from utils.blocking_algo import map_symmetric_neighbors, get_distinct_colors, get_user_input_unpaired

def view_molecule(mol_id, e_centroids, mol_meshes, 
                  neighbor_candidates, neighbor_pairs,
                  view_unpaired_mols=None,
                  distinct_color=False,
                  alpha_value=255):
    """
    Function to visualize a molecule with its mesh, e-centroid, centroid, and neighbors.
    
    Parameters
    ----------
    mol_id: int
        The ID of the molecule to visualize.
    e_centroids : dict[int, np.ndarray]
        Dictionary where keys are molecule IDs (int) and values are
        the corresponding electron clump centroids (numpy arrays of shape (3,)).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary where keys are molecule IDs (int) and values are
        the corresponding molecular meshes (trimesh.Trimesh objects).
    neighbor_candidates : list[tuple[int, int]]
        List of candidate neighbor pairs (i, j) to test for blocking.
    neighbor_pairs : list[tuple[int, int]]
        List of unblocked neighbor pairs (i, j).
    view_unpaired_mols: boolean, int, list[int], None, optional
        List of molecule IDs to specifically view. 
            - True, view all unpaired neighbors.
            - False, view no unpaired neighbors.
            - int, view specified number of unpaired neighbors.
            - list, view specific unpaired neighbors.
            - None, prompt user for input.
    distinct_color: boolean, optional
        If True, use a distinct color scheme for the meshes for higher contrast.
    Returns
    -------
    IPython.core.display.HTML
        A Trimesh scene object for visualization.
    """
    # Parameters
    snc = map_symmetric_neighbors(neighbor_candidates, e_centroids, fill_no_neighbor_mols=True)
    snp = map_symmetric_neighbors(neighbor_pairs, e_centroids, fill_no_neighbor_mols=True)
    snc_ids, snp_ids = snc[mol_id], snp[mol_id]
    exclusive_ids = sorted(list(set(snc_ids).difference(set(snp_ids))))
    
    # Show to user paired neighbors
    print(f'Neighbor pairs for molecule {mol_id}: {snp_ids}')
    print(f'Non-paired neighbors are: {exclusive_ids}\n{"-"*40}')
    time.sleep(1)
    
    # -----------------------------
    # User WANT TO SEE ALL (True), show all unpaired neighbors 
    if view_unpaired_mols is True: user_select_ids = exclusive_ids

    # User DOESN'T WANT TO SEE ANY (False), show no unpaired neighbors
    elif view_unpaired_mols is False: user_select_ids = []
    
    # User WANT TO A FEW, show the number of unpaired neighbors specified by user
    elif isinstance(view_unpaired_mols, int): 
        if view_unpaired_mols < 0:
            print("Please enter a non-negative integer for the number of unpaired neighbors to view."); return None
        elif view_unpaired_mols > len(exclusive_ids):
            print(f"Only {len(exclusive_ids)} unpaired neighbors available. Showing all of them.")
            user_select_ids = exclusive_ids
        else:
            user_select_ids = exclusive_ids[:view_unpaired_mols]
    
    # User DOES NOT SPECIFY (None), prompt user which unpaired neighbors to view
    elif view_unpaired_mols is None: user_select_ids = get_user_input_unpaired(exclusive_ids)
    
    # User DOES SPECIFY (list), use their selection of unpaired neighbors
    else:
        user_select_ids = view_unpaired_mols
        # Check if user input is valid
        if not set(user_select_ids).issubset(set(exclusive_ids)): 
            print("Selected molecule ID not in neighbor list. Please try again."); return None
    # -----------------------------

    # Verified successful input
    print(f"You have selected: {user_select_ids}"); time.sleep(1) 
    
    # Overall molecule meshes to show
    meshes_to_show_ids = sorted([mol_id] + snp_ids + user_select_ids)
    meshes_to_show = [mol_meshes[ids] for ids in meshes_to_show_ids]
    print(f'{'-'*40}\nYou are now viewing molecules {meshes_to_show_ids}')

    # Line mesh for unblocked neighbors
    meshes_lines = []
    target_coord = e_centroids[mol_id]
    for ids in snp_ids:
        neighbor_coord = e_centroids[ids]
        segment = [target_coord, neighbor_coord]
        cyl = trimesh.creation.cylinder(radius=0.3, segment=segment, sections=24)
        cyl.visual.face_colors = [0, 0, 0, 255] # Yellow color
        meshes_lines.append(cyl)
        
    # Palette color scheme for higher contrast
    if distinct_color:
        # alpha_value = 120  # roughly 50% transparent
        palette = get_distinct_colors(len(meshes_to_show), alpha=alpha_value, method='hsv')
        for mesh, color in zip(meshes_to_show, palette):
            color = np.array(color)
            color[3] = alpha_value  # enforce alpha
            mesh.visual.face_colors = np.tile(color, (len(mesh.faces), 1))

    
    meshes = trimesh.util.concatenate(meshes_to_show  + meshes_lines)
    scene = trimesh.Scene(meshes)
    
    return scene.show(viewer='gl')


In [13]:
# PROBLEM: Molecule does not consider itself when blocking neighbors
mol_id = 10 # molecule ID to visualize

view_molecule(mol_id, e_centroids, mol_meshes, neighbor_candidates, 
              neighbor_pairs, view_unpaired_mols=False, distinct_color=True,alpha_value=100)


Neighbor pairs for molecule 10: [58, 125, 418, 587, 1051, 1153]
Non-paired neighbors are: [266, 316, 623, 901]
----------------------------------------
You have selected: []
----------------------------------------
You are now viewing molecules [10, 58, 125, 418, 587, 1051, 1153]


2025-10-13 00:17:14.834 python[11388:28841889] +[IMKClient subclass]: chose IMKClient_Modern
2025-10-13 00:17:14.900 python[11388:28841889] +[IMKInputSession subclass]: chose IMKInputSession_Modern


: 

In [1]:
import trimesh

meshes = []
coords = [[0, 0, 0], [1.5, 0, 0], [0.75, 1.2, 0]]
for pos in coords:
    sph = trimesh.creation.icosphere(radius=1.0)
    sph.apply_translation(pos)
    meshes.append(sph)

# Option 1: simple concatenation
mol_concat = trimesh.util.concatenate(meshes)

# Option 2: watertight union (requires backend)
# mol_union = trimesh.boolean.union(meshes, engine='igl')

mol_concat.show(viewer='gl')   # multi-sphere
# mol_union.show(viewer='glTF')  # single smooth solid


2025-10-13 08:49:11.307 python[12745:28996096] +[IMKClient subclass]: chose IMKClient_Modern
2025-10-13 08:49:11.347 python[12745:28996096] +[IMKInputSession subclass]: chose IMKInputSession_Modern


: 